## Установка зависимостей

In [40]:
!pip install -q fastapi "uvicorn[standard]" nest_asyncio scikit-learn pandas requests joblib

## Обучим простую модель (Iris) и сохраним её на диск (model.joblib)

**Описание:**

Загружает датасет Iris из sklearn.datasets.

Делит данные на train/test (80/20).

Обучает LogisticRegression (простой классификатор).

Сохраняет в model.joblib словарь с ключами 'model', 'feature_names', 'target_names' через joblib.dump.

In [41]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import joblib
import numpy as np

In [42]:
data = load_iris()
X = data['data']  # 4 признака
y = data['target']
feature_names = data['feature_names']
target_names = data['target_names']

In [43]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

LogisticRegression(max_iter=200)

In [44]:
joblib.dump({
    'model': model,
    'feature_names': feature_names,
    'target_names': list(target_names)
}, 'model.joblib')

# Проверка
acc = model.score(X_test, y_test)
print(f"Модель обучена. Точность на тесте: {acc:.3f}")
print("Сохранено в model.joblib")

Модель обучена. Точность на тесте: 1.000
Сохранено в model.joblib


## FastAPI приложение: CRUD для списка покупок, /expenses и /predict

Создаёт FastAPI-приложение с:

- Метаданными приложения (title, description, version, contact, license_info) — влияет на OpenAPI / Swagger UI.

- Pydantic-моделями: ItemCreate, Item, ExpensePerGroup, ExpensesResponse, PredictRequest, PredictResponse.

- CRUD-эндпоинтами для /items/ (POST, GET), /items/{id} (GET/PUT/DELETE).

- GET /expenses — собирает расходы по группам и возвращает общий итог.

- POST /predict — применяет загруженную модель для входных признаков.

In [45]:
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel, Field
from typing import Optional, List, Dict
import uvicorn
import joblib
from decimal import Decimal
from uuid import uuid4

### Загрузка модели

In [46]:
saved = joblib.load('model.joblib')
sk_model = saved['model']
MODEL_FEATURES = saved['feature_names']
MODEL_TARGETS = saved['target_names']

app = FastAPI(
    title="Shopping List & Model API",
    description=(
        "REST API для списка покупок (CRUD). "
        "Также endpoint `/expenses` для суммирования расходов по группам и общей суммы. "
        "Endpoint `/predict` применяет простую модель (Iris) для демонстрации."
    ),
    version="1.0.0",
    contact={
        "name": "Student",
        "email": "student@example.com",
    },
    license_info={
        "name": "MIT"
    }
)

### Pydantic модели

In [47]:
class ItemCreate(BaseModel):
    name: str = Field(..., example="Milk", description="Название товара")
    group: str = Field(..., example="Продовольствие", description="Группа товара")
    price: float = Field(..., gt=0, example=1.25, description="Цена за единицу")
    unit: str = Field(..., example="L", description="Единица измерения")
    quantity: float = Field(..., gt=0, example=2, description="Количество единиц")

In [48]:
class Item(ItemCreate):
    id: str = Field(..., example="b2f1e5a0", description="Уникальный ID товара")

In [49]:
class ExpensePerGroup(BaseModel):
    group: str
    expense: float

In [50]:
class ExpensesResponse(BaseModel):
    per_group: List[ExpensePerGroup]
    total: float

In [51]:
class PredictRequest(BaseModel):
    features: List[float] = Field(..., description=f"Список признаков. Ожидаемые признаки: {MODEL_FEATURES}", example=[5.1, 3.5, 1.4, 0.2])

In [52]:
class PredictResponse(BaseModel):
    predicted_class: str
    predicted_index: int
    probabilities: List[float]

### "База данных" в памяти

In [53]:
ITEMS: Dict[str, Item] = {}

### CRUD endpoints

In [54]:
@app.post("/items/", response_model=Item, status_code=status.HTTP_201_CREATED, tags=["items"], summary="Create item", response_description="Созданный объект товара")
def create_item(item: ItemCreate):
    item_id = uuid4().hex[:8]
    new = Item(id=item_id, **item.dict())
    ITEMS[item_id] = new
    return new

In [55]:
@app.get("/items/", response_model=List[Item], tags=["items"], summary="List items", response_description="Список всех товаров")
def list_items():
    return list(ITEMS.values())

In [56]:
@app.get("/items/{item_id}", response_model=Item, tags=["items"], summary="Get item by id")
def get_item(item_id: str):
    item = ITEMS.get(item_id)
    if not item:
        raise HTTPException(status_code=404, detail="Item not found")
    return item

In [57]:
@app.put("/items/{item_id}", response_model=Item, tags=["items"], summary="Update item")
def update_item(item_id: str, item: ItemCreate):
    existing = ITEMS.get(item_id)
    if not existing:
        raise HTTPException(status_code=404, detail="Item not found")
    updated = Item(id=item_id, **item.dict())
    ITEMS[item_id] = updated
    return updated

In [58]:
@app.delete("/items/{item_id}", status_code=status.HTTP_204_NO_CONTENT, tags=["items"], summary="Delete item")
def delete_item(item_id: str):
    if item_id in ITEMS:
        del ITEMS[item_id]
        return
    raise HTTPException(status_code=404, detail="Item not found")

### Expenses endpoint

In [59]:
@app.get("/expenses", response_model=ExpensesResponse, tags=["analytics"], summary="Expenses per group and total", response_description="Возвращает список расходов по каждой группе и общую сумму")
def expenses():
    group_sums: Dict[str, float] = {}
    total = 0.0
    for it in ITEMS.values():
        cost = float(it.price) * float(it.quantity)
        group_sums[it.group] = group_sums.get(it.group, 0.0) + cost
        total += cost
    per_group = [ExpensePerGroup(group=k, expense=v) for k, v in group_sums.items()]
    return ExpensesResponse(per_group=per_group, total=total)

### Predict endpoint using sklearn model

In [60]:
@app.post("/predict", response_model=PredictResponse, tags=["model"], summary="Predict using trained model", response_description="Предсказание класса и вероятности")
def predict(req: PredictRequest):
    x = req.features
    if len(x) != len(MODEL_FEATURES):
        raise HTTPException(status_code=400, detail=f"Expected {len(MODEL_FEATURES)} features: {MODEL_FEATURES}")
    import numpy as np
    arr = np.array(x).reshape(1, -1)
    pred_index = int(sk_model.predict(arr)[0])
    probs = sk_model.predict_proba(arr)[0].tolist()
    return PredictResponse(predicted_class=MODEL_TARGETS[pred_index], predicted_index=pred_index, probabilities=probs)

### Root

In [61]:
@app.get("/", tags=["root"], summary="Root")
def read_root():
    return {"message": "Shopping List & Model API. See /docs for interactive documentation."}

## Запуск uvicorn в отдельном потоке

Поднимает локальный uvicorn-сервер на 127.0.0.1:8000 в фоне (потоке), чтобы в том же ноутбуке можно было отправлять HTTP-запросы к нему. Используется nest_asyncio для корректной работы event loop в Colab.

In [62]:
import nest_asyncio, threading, time
nest_asyncio.apply()

def _run():
    uvicorn.run("main:app" if False else app, host="127.0.0.1", port=8000, log_level="info")  # используем app из текущего namespace

thread = threading.Thread(target=_run, daemon=True)
thread.start()
time.sleep(1)  # небольшая пауза чтобы сервер успел подняться
print("Server started on http://127.0.0.1:8000 — открой /docs для тестирования")


INFO:     Started server process [187]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


Server started on http://127.0.0.1:8000 — открой /docs для тестирования


## Демонстрация всех методов с помощью библиотеки requests

Покрывает «тестами» (демонстрационными запросами) все реализованные методы API — это имитация тестового покрытия (как у QA).
Выполняются запросы:

GET / — проверка сервера.

POST /items/ — создание нескольких записей.

GET /items/ — список.

GET /items/{id} — получить один элемент.

PUT /items/{id} — обновить.

GET /expenses — получить расходы по группам и общую сумму.

DELETE /items/{id} — удалить элемент.

POST /predict — предсказание модели.

In [63]:
import requests
BASE = "http://127.0.0.1:8000"

print("Root:", requests.get(BASE + "/").json())

INFO:     127.0.0.1:52628 - "GET / HTTP/1.1" 200 OK
Root: {'message': 'Shopping List & Model API. See /docs for interactive documentation.'}


## 1) Создадим несколько товаров

In [64]:
items_to_create = [
    {"name":"Milk", "group":"Продовольствие", "price":0.99, "unit":"L", "quantity":2},
    {"name":"Bread", "group":"Продовольствие", "price":1.50, "unit":"pcs", "quantity":1},
    {"name":"USB Cable", "group":"Электроника", "price":5.99, "unit":"pcs", "quantity":1},
    {"name":"Apples", "group":"Продовольствие", "price":0.5, "unit":"kg", "quantity":3},
]

created = []
for it in items_to_create:
    r = requests.post(BASE + "/items/", json=it)
    r.raise_for_status()
    created_item = r.json()
    created.append(created_item)
    print("Created:", created_item)

INFO:     127.0.0.1:52632 - "POST /items/ HTTP/1.1" 201 Created
Created: {'name': 'Milk', 'group': 'Продовольствие', 'price': 0.99, 'unit': 'L', 'quantity': 2.0, 'id': '9f2e81f9'}
INFO:     127.0.0.1:52642 - "POST /items/ HTTP/1.1" 201 Created
Created: {'name': 'Bread', 'group': 'Продовольствие', 'price': 1.5, 'unit': 'pcs', 'quantity': 1.0, 'id': 'c995c5d2'}
INFO:     127.0.0.1:52644 - "POST /items/ HTTP/1.1" 201 Created
Created: {'name': 'USB Cable', 'group': 'Электроника', 'price': 5.99, 'unit': 'pcs', 'quantity': 1.0, 'id': 'b6a7a0d6'}
INFO:     127.0.0.1:52648 - "POST /items/ HTTP/1.1" 201 Created
Created: {'name': 'Apples', 'group': 'Продовольствие', 'price': 0.5, 'unit': 'kg', 'quantity': 3.0, 'id': 'f1a18502'}


/tmp/ipython-input-2951087682.py:73: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  new = Item(id=item_id, **item.dict())


## 2) Список всех товаров

In [65]:
all_items = requests.get(BASE + "/items/").json()
print("\nAll items:", all_items)

INFO:     127.0.0.1:52650 - "GET /items/ HTTP/1.1" 200 OK

All items: [{'name': 'Milk', 'group': 'Продовольствие', 'price': 0.99, 'unit': 'L', 'quantity': 2.0, 'id': '9f2e81f9'}, {'name': 'Bread', 'group': 'Продовольствие', 'price': 1.5, 'unit': 'pcs', 'quantity': 1.0, 'id': 'c995c5d2'}, {'name': 'USB Cable', 'group': 'Электроника', 'price': 5.99, 'unit': 'pcs', 'quantity': 1.0, 'id': 'b6a7a0d6'}, {'name': 'Apples', 'group': 'Продовольствие', 'price': 0.5, 'unit': 'kg', 'quantity': 3.0, 'id': 'f1a18502'}]


## 3) Получение одного товара по id

In [66]:
item_id = created[0]['id']
print("\nGet item:", requests.get(f"{BASE}/items/{item_id}").json())

INFO:     127.0.0.1:52654 - "GET /items/9f2e81f9 HTTP/1.1" 200 OK

Get item: {'name': 'Milk', 'group': 'Продовольствие', 'price': 0.99, 'unit': 'L', 'quantity': 2.0, 'id': '9f2e81f9'}


## 4) Обновление товара

In [67]:
update_payload = {"name":"Milk (Semi-skimmed)", "group":"Продовольствие", "price":1.05, "unit":"L", "quantity":2}
print("\nUpdate response:", requests.put(f"{BASE}/items/{item_id}", json=update_payload).json())

INFO:     127.0.0.1:52664 - "PUT /items/9f2e81f9 HTTP/1.1" 200 OK

Update response: {'name': 'Milk (Semi-skimmed)', 'group': 'Продовольствие', 'price': 1.05, 'unit': 'L', 'quantity': 2.0, 'id': '9f2e81f9'}


/tmp/ipython-input-2951087682.py:93: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  updated = Item(id=item_id, **item.dict())


## 5) Expenses (расходы по группам и общая сумма)

In [68]:
exp = requests.get(BASE + "/expenses").json()
print("\nExpenses:", exp)

INFO:     127.0.0.1:52668 - "GET /expenses HTTP/1.1" 200 OK

Expenses: {'per_group': [{'group': 'Продовольствие', 'expense': 5.1}, {'group': 'Электроника', 'expense': 5.99}], 'total': 11.09}


## 6) Удаление товара

In [69]:
to_delete = created[2]['id']
del_resp = requests.delete(f"{BASE}/items/{to_delete}")
print(f"\nDelete {to_delete} status:", del_resp.status_code)

INFO:     127.0.0.1:52670 - "DELETE /items/b6a7a0d6 HTTP/1.1" 204 No Content

Delete b6a7a0d6 status: 204


## 7) Снова список

In [70]:
print("\nAll items after delete:", requests.get(BASE + "/items/").json())

INFO:     127.0.0.1:52676 - "GET /items/ HTTP/1.1" 200 OK

All items after delete: [{'name': 'Milk (Semi-skimmed)', 'group': 'Продовольствие', 'price': 1.05, 'unit': 'L', 'quantity': 2.0, 'id': '9f2e81f9'}, {'name': 'Bread', 'group': 'Продовольствие', 'price': 1.5, 'unit': 'pcs', 'quantity': 1.0, 'id': 'c995c5d2'}, {'name': 'Apples', 'group': 'Продовольствие', 'price': 0.5, 'unit': 'kg', 'quantity': 3.0, 'id': 'f1a18502'}]


## 8) Predict — пример с признаками Iris

In [71]:
predict_payload = {"features": [5.1, 3.5, 1.4, 0.2]}
pred = requests.post(BASE + "/predict", json=predict_payload)
print("\nPredict response:", pred.json())

INFO:     127.0.0.1:52680 - "POST /predict HTTP/1.1" 200 OK

Predict response: {'predicted_class': 'setosa', 'predicted_index': 0, 'probabilities': [0.9765541100591919, 0.02344584100247694, 4.893833125239376e-08]}


# Краткая сводка результатов

- Модель обучена и сохранена: model.joblib. Точность на тесте: 1.000.

- Сервер FastAPI успешно поднят на http://127.0.0.1:8000.

- Выполнены демонстрационные запросы — создано 4 товара (в примере): Milk, Bread, USB Cable, Apples (сгенерированные id: b9ce265c, c92689b6, 787f0572, 25cc4a8f в вашем логе).

- /expenses вернул:

In [ ]:
{'per_group': [{'group': 'Продовольствие', 'expense': 5.1}, {'group': 'Электроника', 'expense': 5.99}], 'total': 11.09}

- Удаление USB Cable вернуло 204 No Contentv

- /predict вернул предсказание (пример):

In [ ]:
{'predicted_class': 'setosa', 'predicted_index': 0, 'probabilities': [0.9765541100591919, 0.02344584100247694, 4.893833125239376e-08]}